# PredictionDriftDetector – Usage Examples

This notebook demonstrates how to use the `PredictionDriftDetector` class to monitor drift in model prediction outputs between a reference dataset and a current dataset.

**Features:**
- **Continuous predictions** (e.g., regression scores, probabilities): summary statistics (mean, std, quantiles), histogram distances (L1/L2), PSI, KS, KL, JS, and decile shifts.
- **Categorical predictions** (e.g., class labels): category proportions, PSI, KL, JS (KS and histogram distances not applicable).
- Automatic detection of prediction type based on data and configurable threshold.
- Optional inclusion of KS, KL, JS metrics.
- Customizable binning, quantiles, and handling of values outside reference bins.

All metrics are computed for both reference and current datasets, and deltas are provided.

In [1]:
import numpy as np
import pandas as pd
import warnings

# The class we want to demonstrate
from momo_ml.monitor.prediction_drift import PredictionDriftDetector

# For reproducibility
np.random.seed(42)

In [2]:
# Reference dataset (e.g., training predictions)
n_ref = 1000
ref_continuous = pd.DataFrame({
    'pred': np.random.normal(loc=0.5, scale=0.2, size=n_ref)  # probabilities around 0.5
})

# Current dataset – with a shift (e.g., production predictions)
n_cur = 800
cur_continuous = pd.DataFrame({
    'pred': np.random.normal(loc=0.6, scale=0.25, size=n_cur)  # mean shifted to 0.6
})

print("Reference shape:", ref_continuous.shape)
print("Current shape:", cur_continuous.shape)
ref_continuous.head()

Reference shape: (1000, 1)
Current shape: (800, 1)


,pred
0,0.599343
1,0.472347
2,0.629538
3,0.804606
4,0.453169


## 1. Basic Continuous Prediction Drift Detection

In [8]:
# Initialize detector with default settings (PSI enabled, KS/KL/JS disabled)
detector = PredictionDriftDetector(
    ref_df=ref_continuous,
    cur_df=cur_continuous,
    pred_col='pred'
)

# Compute drift
result = detector.compute()

print("Prediction type:", result['prediction_type'])
print("\n--- Summary Statistics ---")
for stat, values in result['summary_statistics'].items():
    if isinstance(values, dict):
        if 'delta' in values:
            print(f"{stat}: ref={values['reference']:.4f}, cur={values['current']:.4f}, delta={values['delta']:.4f}")
        else:
            print(f"{stat}: ref={values['reference']:.4f}, cur={values['current']:.4f}")
    else:
        print(f"{stat}: {values}")

print("\n--- Distribution Shift ---")
for metric, value in result['distribution_shift'].items():
    if metric == 'ks':
        print(f"{metric}: statistic={value['statistic']:.4f}, pvalue={value['pvalue']:.4f}")
    else:
        print(f"{metric}: {value:.4f}")

print("\n--- Decile Shift (first 5) ---")
dec = result['decile_shift']
print("Quantiles:", dec['quantiles'][:5])
print("Ref values:", dec['ref_values'][:5])
print("Cur values:", dec['cur_values'][:5])
print("Delta:", dec['delta'][:5])

Prediction type: continuous

--- Summary Statistics ---
mean: ref=0.5039, cur=0.6210, delta=0.1171
std: ref=0.1957, cur=0.2500, delta=0.0542
min: ref=-0.1483, cur=-0.1303
max: ref=1.2705, cur=1.3983
q25: ref=0.3705, cur=0.4486
q50: ref=0.5051, cur=0.6172
q75: ref=0.6296, cur=0.7849

--- Distribution Shift ---
l1_distance: 6.0070
l2_distance: 1.7491
psi: 0.3690

--- Decile Shift (first 5) ---
Quantiles: [0.0, 0.1, 0.2, 0.30000000000000004, 0.4]
Ref values: [-0.14825346801381456, 0.2510473778297052, 0.3393037055103349, 0.39532838022175604, 0.4518620675925174]
Cur values: [-0.13033762087365186, 0.3050260865133066, 0.4129681116656761, 0.49248250684922706, 0.5576582792690056]
Delta: [0.0179158471401627, 0.053978708683601384, 0.0736644061553412, 0.09715412662747103, 0.10579621167648817]


In [9]:
# The result is a dictionary with clear sections
print("Top-level keys:", result.keys())
print("\nSummary statistics keys:", result['summary_statistics'].keys())
print("Distribution shift keys:", result['distribution_shift'].keys())
print("Decile shift keys:", result['decile_shift'].keys())

Top-level keys: dict_keys(['prediction_type', 'summary_statistics', 'distribution_shift', 'decile_shift'])

Summary statistics keys: dict_keys(['mean', 'std', 'min', 'max', 'q25', 'q50', 'q75'])
Distribution shift keys: dict_keys(['l1_distance', 'l2_distance', 'psi'])
Decile shift keys: dict_keys(['quantiles', 'ref_values', 'cur_values', 'delta'])


## 2. Including Optional Metrics (KS, KL, JS)

In [10]:
# Enable all optional metrics
detector_full = PredictionDriftDetector(
    ref_continuous, cur_continuous,
    pred_col='pred',
    include_ks=True,
    include_kl=True,
    include_js=True,
    kl_base='2',          # use bits for KL/JS
    kl_epsilon=1e-10,
    kl_handle_outside='clip'   # clip values outside reference bins
)

result_full = detector_full.compute()

print("--- Distribution Shift with All Metrics ---")
for metric, value in result_full['distribution_shift'].items():
    if metric == 'ks':
        print(f"{metric}: statistic={value['statistic']:.4f}, pvalue={value['pvalue']:.4f}")
    else:
        print(f"{metric}: {value:.4f}")

--- Distribution Shift with All Metrics ---
l1_distance: 6.0070
l2_distance: 1.7491
psi: 0.3690
ks: statistic=0.2407, pvalue=0.0000
kl: 0.2430
js: 0.0637


## 3. Generate Sample Data – Categorical Predictions

In [11]:
# Reference: class labels with certain proportions
ref_cat = pd.DataFrame({
    'pred': np.random.choice(['low', 'medium', 'high'], size=1000, p=[0.5, 0.3, 0.2])
})

# Current: shifted proportions
cur_cat = pd.DataFrame({
    'pred': np.random.choice(['low', 'medium', 'high'], size=800, p=[0.4, 0.4, 0.2])
})

print("Reference value counts:")
print(ref_cat['pred'].value_counts(normalize=True))
print("\nCurrent value counts:")
print(cur_cat['pred'].value_counts(normalize=True))

Reference value counts:
pred
low       0.509
medium    0.300
high      0.191
Name: proportion, dtype: float64

Current value counts:
pred
low       0.41125
medium    0.38125
high      0.20750
Name: proportion, dtype: float64


## 4. Categorical Prediction Drift Detection

In [12]:
detector_cat = PredictionDriftDetector(
    ref_cat, cur_cat,
    pred_col='pred',
    include_kl=True,
    include_js=True
)

result_cat = detector_cat.compute()

print("Prediction type:", result_cat['prediction_type'])
print("\n--- Summary Statistics (Category Proportions) ---")
stats = result_cat['summary_statistics']
print("Categories:", stats['categories'])
print("Reference proportions:", stats['reference_proportions'])
print("Current proportions:", stats['current_proportions'])
print("Delta proportions:", stats['delta_proportions'])

print("\n--- Distribution Shift (Categorical) ---")
dist = result_cat['distribution_shift']
print("PSI:", dist.get('psi', 'N/A'))
print("KL:", dist.get('kl', 'N/A'))
print("JS:", dist.get('js', 'N/A'))
# Note: histogram distances and KS are not present for categorical data

Prediction type: categorical

--- Summary Statistics (Category Proportions) ---
Categories: ['high', 'low', 'medium']
Reference proportions: {'high': np.float64(0.191), 'low': np.float64(0.509), 'medium': np.float64(0.3)}
Current proportions: {'high': np.float64(0.2075), 'low': np.float64(0.41125), 'medium': np.float64(0.38125)}
Delta proportions: {'high': np.float64(0.016499999999999987), 'low': np.float64(-0.09775), 'medium': np.float64(0.08124999999999999)}

--- Distribution Shift (Categorical) ---
PSI: 0.041685441209147925
KL: 0.020814860565875462
JS: 0.005199896058266488


## 5. Customizing Parameters

In [13]:
# Example: change number of bins, quantiles, and threshold for categorical detection
detector_custom = PredictionDriftDetector(
    ref_continuous, cur_continuous,
    pred_col='pred',
    bins=50,                      # finer histogram bins
    quantiles=[0.0, 0.25, 0.5, 0.75, 1.0],  # custom quantiles
    categorical_threshold=10,     # if unique values ≤10, treat as categorical
    include_psi=True
)

result_custom = detector_custom.compute()
print("Custom quantiles:", result_custom['decile_shift']['quantiles'])
print("Custom binning – L1 distance:", result_custom['distribution_shift']['l1_distance'])

Custom quantiles: [0.0, 0.25, 0.5, 0.75, 1.0]
Custom binning – L1 distance: 15.45394815293887


## 6. Handling Missing Columns or Empty Data

In [14]:
# Missing prediction column
ref_missing = ref_continuous.copy()
cur_missing = cur_continuous.copy()
detector_missing = PredictionDriftDetector(ref_missing, cur_missing, pred_col='nonexistent')
result_missing = detector_missing.compute()
print("Error when column missing:", result_missing.get('error'))

# All NaN values
ref_nan = pd.DataFrame({'pred': [np.nan]*100})
cur_nan = pd.DataFrame({'pred': [np.nan]*80})
detector_nan = PredictionDriftDetector(ref_nan, cur_nan, pred_col='pred')
result_nan = detector_nan.compute()
print("Error when all NaN:", result_nan.get('error'))

Error when column missing: Column 'nonexistent' not found in both datasets.
Error when all NaN: No valid prediction values found.


The `PredictionDriftDetector` provides a comprehensive tool for monitoring shifts in model predictions between two datasets. Key takeaways:

- **Automatic type detection**: Distinguishes between continuous and categorical predictions based on data and a threshold.
- **Rich set of metrics**: PSI (always available), optional KS, KL, JS, and histogram distances for continuous data.
- **Structured output**: Returns a dictionary with `prediction_type`, `summary_statistics`, `distribution_shift`, and (for continuous) `decile_shift`.
- **Flexible configuration**: Control binning, quantiles, handling of out-of-bounds values, and which metrics to compute.
- **Robust error handling**: Returns descriptive error messages when columns are missing or data is empty.

These metrics help data scientists and ML engineers detect when model behaviour changes in production, enabling timely interventions.